# 07 — 100-Image / 4-Model PC + Android Benchmark

Notebook này gom toàn bộ flow mới:

1. Kiểm tra 4 model TFLite trong `models/deploy`.
2. Copy 4 model sang Android `app/src/main/assets/models/`.
3. Chọn cố định **100 ảnh** từ test set theo hướng tăng độ phủ class.
4. Copy 100 ảnh sang Android `assets/benchmark_images/`.
5. Tạo subset dataset 100 ảnh trên PC để đánh giá cả 4 model bằng cùng tập ảnh.
6. Chạy 4 TFLite model trên PC và tạo bảng Precision / Recall / mAP50 / mAP50-95.
7. Sau khi Android chạy một nút cho cả 4 model, đọc CSV Android và ghép vào cùng bảng.
8. Xuất master comparison table để làm trade-off.

> 100 ảnh này là **controlled subset** để so PC ↔ Android và so 4 model trong cùng điều kiện.  
> Accuracy chính thức của model vẫn ưu tiên full test set 1151 ảnh đã đo trước đó.


## 1. Cấu hình đường dẫn


In [ ]:
from pathlib import Path
import shutil
import numpy as np
import pandas as pd
import yaml

ROOT = Path(r"D:\Project\TrafficSignAI")
ANDROID_ROOT = Path(r"D:\Project\AndroidProjects\TrafficSignApp")

DATASET_ROOT = ROOT / "VR-TSD-2"
TEST_IMAGES = DATASET_ROOT / "test" / "images"
TEST_LABELS = DATASET_ROOT / "test" / "labels"

DEPLOY_DIR = ROOT / "models" / "deploy"

ANDROID_ASSET_ROOT = ANDROID_ROOT / "app" / "src" / "main" / "assets"
ANDROID_MODEL_DIR = ANDROID_ASSET_ROOT / "models"
ANDROID_IMAGE_DIR = ANDROID_ASSET_ROOT / "benchmark_images"

BENCHMARK_ROOT = ROOT / "android_benchmark_100"
PC_SUBSET_ROOT = BENCHMARK_ROOT / "pc_subset_dataset"
ANDROID_RESULT_DIR = BENCHMARK_ROOT / "android_results"

BENCHMARK_ROOT.mkdir(parents=True, exist_ok=True)
ANDROID_RESULT_DIR.mkdir(parents=True, exist_ok=True)

MODELS = {
    "yolo11n_320": {
        "file": "yolo11n_320.tflite",
        "input_size": 320,
    },
    "yolo11n_640": {
        "file": "yolo11n_640.tflite",
        "input_size": 640,
    },
    "yolo11s_320": {
        "file": "yolo11s_320.tflite",
        "input_size": 320,
    },
    "yolo11s_640": {
        "file": "yolo11s_640.tflite",
        "input_size": 640,
    },
}

print("DEPLOY_DIR:", DEPLOY_DIR)
print("ANDROID_MODEL_DIR:", ANDROID_MODEL_DIR)
print("ANDROID_IMAGE_DIR:", ANDROID_IMAGE_DIR)
print("BENCHMARK_ROOT:", BENCHMARK_ROOT)


## 2. Kiểm tra và copy 4 model sang Android

Sau cell này, Android phải có:

```text
app/src/main/assets/
├── labels.txt
├── models/
│   ├── yolo11n_320.tflite
│   ├── yolo11n_640.tflite
│   ├── yolo11s_320.tflite
│   └── yolo11s_640.tflite
└── benchmark_images/
```


In [ ]:
ANDROID_MODEL_DIR.mkdir(parents=True, exist_ok=True)

missing = []

for model_id, cfg in MODELS.items():
    src = DEPLOY_DIR / cfg["file"]

    if not src.exists():
        missing.append(src)
        continue

    dst = ANDROID_MODEL_DIR / cfg["file"]
    shutil.copy2(src, dst)

    print(
        f"{model_id}: {src.stat().st_size / 1024**2:.2f} MiB "
        f"→ {dst}"
    )

if missing:
    raise FileNotFoundError(
        "Thiếu model:\n" +
        "\n".join(str(x) for x in missing)
    )

print("\nĐã copy đủ 4 model sang Android assets/models.")


## 3. Chọn 100 ảnh theo hướng tăng độ phủ class

Để bộ 100 ảnh ít phụ thuộc vào random thuần:

- đầu tiên ưu tiên ảnh giúp bổ sung các class chưa được chọn;
- sau khi không tăng thêm class coverage nữa thì fill ngẫu nhiên bằng seed cố định;
- giữ nguyên filename;
- cùng 100 ảnh này sẽ được dùng cho PC và Android.

Cách này phù hợp cho controlled comparison hơn random 100 ảnh đơn thuần.


In [ ]:
SEED = 42
NUM_IMAGES = 100

def read_classes(label_path: Path):
    if not label_path.exists():
        return set()

    classes = set()

    for line in label_path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line:
            continue
        classes.add(int(line.split()[0]))

    return classes

records = []

for image_path in sorted(TEST_IMAGES.iterdir()):
    if image_path.suffix.lower() not in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}:
        continue

    label_path = TEST_LABELS / f"{image_path.stem}.txt"
    classes = read_classes(label_path)

    if not classes:
        continue

    records.append({
        "image": image_path.name,
        "image_path": image_path,
        "label_path": label_path,
        "classes": classes,
        "object_count": len([
            x for x in label_path.read_text(encoding="utf-8").splitlines()
            if x.strip()
        ])
    })

rng = np.random.default_rng(SEED)

selected = []
remaining = records.copy()

all_classes = set().union(*(r["classes"] for r in records))
covered = set()

# Greedy class coverage.
while remaining and len(selected) < NUM_IMAGES:
    gains = np.array([
        len(r["classes"] - covered)
        for r in remaining
    ])

    max_gain = gains.max()

    if max_gain <= 0:
        break

    candidate_ids = np.flatnonzero(gains == max_gain)
    chosen_idx = int(rng.choice(candidate_ids))

    chosen = remaining.pop(chosen_idx)
    selected.append(chosen)
    covered |= chosen["classes"]

# Fill phần còn lại bằng random cố định.
if len(selected) < NUM_IMAGES:
    need = NUM_IMAGES - len(selected)

    chosen_ids = rng.choice(
        len(remaining),
        size=min(need, len(remaining)),
        replace=False
    )

    for idx in sorted(chosen_ids, reverse=True):
        selected.append(remaining.pop(int(idx)))

selected = selected[:NUM_IMAGES]

selected_df = pd.DataFrame([
    {
        "image": r["image"],
        "object_count": r["object_count"],
        "class_count": len(r["classes"]),
        "class_ids": " ".join(map(str, sorted(r["classes"])))
    }
    for r in selected
])

print("Selected images:", len(selected_df))
print("Class coverage:", len(covered), "/", len(all_classes))
print("Covered class IDs:", sorted(covered))

display(selected_df.head(20))


## 4. Copy 100 ảnh sang Android và tạo PC subset dataset


In [ ]:
# ---------- Android benchmark_images ----------
ANDROID_IMAGE_DIR.mkdir(parents=True, exist_ok=True)

for old in ANDROID_IMAGE_DIR.iterdir():
    if old.is_file():
        old.unlink()

for r in selected:
    shutil.copy2(
        r["image_path"],
        ANDROID_IMAGE_DIR / r["image"]
    )

print("Android benchmark images:", len(list(ANDROID_IMAGE_DIR.iterdir())))

# ---------- PC subset dataset ----------
subset_images = PC_SUBSET_ROOT / "images"
subset_labels = PC_SUBSET_ROOT / "labels"

if PC_SUBSET_ROOT.exists():
    shutil.rmtree(PC_SUBSET_ROOT)

subset_images.mkdir(parents=True, exist_ok=True)
subset_labels.mkdir(parents=True, exist_ok=True)

for r in selected:
    shutil.copy2(
        r["image_path"],
        subset_images / r["image"]
    )

    shutil.copy2(
        r["label_path"],
        subset_labels / r["label_path"].name
    )

# Lấy names/nc từ data_local.yaml nếu có, nếu không dùng data.yaml.
source_yaml = DATASET_ROOT / "data_local.yaml"

if not source_yaml.exists():
    source_yaml = DATASET_ROOT / "data.yaml"

with open(source_yaml, "r", encoding="utf-8") as f:
    source_data = yaml.safe_load(f)

subset_yaml = PC_SUBSET_ROOT / "data_100.yaml"

subset_data = {
    "path": str(PC_SUBSET_ROOT.resolve()),
    "train": "images",
    "val": "images",
    "test": "images",
    "nc": int(source_data["nc"]),
    "names": source_data["names"],
}

with open(subset_yaml, "w", encoding="utf-8") as f:
    yaml.safe_dump(
        subset_data,
        f,
        allow_unicode=True,
        sort_keys=False
    )

manifest = BENCHMARK_ROOT / "benchmark_100_manifest.csv"

selected_df.to_csv(
    manifest,
    index=False,
    encoding="utf-8-sig"
)

print("Subset YAML:", subset_yaml)
print("Manifest:", manifest)
print("Images on Android:", len(list(ANDROID_IMAGE_DIR.iterdir())))


## 5. Full-test reference metrics đã có

Đây là accuracy TFLite trên toàn test set trước đó. Ta giữ để làm chuẩn chính thức.


In [ ]:
full_test = pd.DataFrame([
    {
        "model": "yolo11n_320",
        "full_precision": 0.7805,
        "full_recall": 0.6604,
        "full_mAP50": 0.7297,
        "full_mAP50_95": 0.5778,
        "model_size_mib": 10.166,
    },
    {
        "model": "yolo11n_640",
        "full_precision": 0.9132,
        "full_recall": 0.8322,
        "full_mAP50": 0.9203,
        "full_mAP50_95": 0.7496,
        "model_size_mib": 10.178,
    },
    {
        "model": "yolo11s_320",
        "full_precision": 0.9144,
        "full_recall": 0.8465,
        "full_mAP50": 0.8975,
        "full_mAP50_95": 0.7083,
        "model_size_mib": 36.206,
    },
    {
        "model": "yolo11s_640",
        "full_precision": 0.9352,
        "full_recall": 0.9421,
        "full_mAP50": 0.9704,
        "full_mAP50_95": 0.8064,
        "model_size_mib": 36.278,
    },
])

display(
    full_test.sort_values(
        "full_mAP50_95",
        ascending=False
    )
)


## 6. Đánh giá cả 4 TFLite model trên đúng 100 ảnh ở PC

Kết quả phần này là **subset accuracy**, dùng để so cùng tập 100 ảnh với Android.

Không dùng nó thay cho full-test accuracy phía trên.


In [ ]:
from ultralytics import YOLO

PC_RESULT_CSV = BENCHMARK_ROOT / "pc_100image_4model_metrics.csv"

rows = []

for model_id, cfg in MODELS.items():

    model_path = DEPLOY_DIR / cfg["file"]

    print("\n" + "=" * 80)
    print("PC subset validation:", model_id)
    print("=" * 80)

    model = YOLO(
        str(model_path),
        task="detect"
    )

    metrics = model.val(
        data=str(subset_yaml),
        split="test",
        imgsz=cfg["input_size"],
        batch=1,
        verbose=False,
        plots=False
    )

    rows.append({
        "model": model_id,
        "input_size": cfg["input_size"],
        "pc100_precision": float(metrics.box.mp),
        "pc100_recall": float(metrics.box.mr),
        "pc100_mAP50": float(metrics.box.map50),
        "pc100_mAP50_95": float(metrics.box.map),
    })

pc100 = pd.DataFrame(rows)

pc100.to_csv(
    PC_RESULT_CSV,
    index=False,
    encoding="utf-8-sig"
)

display(
    pc100.sort_values(
        "pc100_mAP50_95",
        ascending=False
    )
)

print("Saved:", PC_RESULT_CSV)


## 7. Chuẩn bị Android

Sau khi chạy các cell trên, Android project phải có:

```text
app/src/main/assets/
├── labels.txt
├── models/
│   ├── yolo11n_320.tflite
│   ├── yolo11n_640.tflite
│   ├── yolo11s_320.tflite
│   └── yolo11s_640.tflite
└── benchmark_images/
    └── 100 ảnh
```

Sau đó replace các file Kotlin phiên bản 4-model rồi Build + Run.

Khi bấm nút `RUN 4-MODEL TEST`, Android sẽ chạy tuần tự:

`n320 → n640 → s320 → s640`

Mỗi model:
- load riêng;
- warm-up 5 lần;
- chạy cùng 100 ảnh;
- đo confidence, latency, CPU, RAM;
- đóng model trước khi sang model kế tiếp.

Điện thoại sẽ xuất 3 CSV:
- `android_4model_100_summary_*.csv` — 4 dòng, một dòng/model.
- `android_4model_100_detail_*.csv` — 400 dòng, một dòng/model/ảnh.
- `android_4model_100_detections_*.csv` — toàn bộ bounding boxes.

Copy 3 file vào:

`D:\Project\TrafficSignAI\android_benchmark_100\android_results\`


## 8. Đọc Android 4-model summary và ghép với PC


In [ ]:
summary_files = sorted(
    ANDROID_RESULT_DIR.glob(
        "android_4model_100_summary_*.csv"
    )
)

if not summary_files:

    print(
        "Chưa có Android result.\n"
        "Sau khi điện thoại chạy xong, copy file "
        "android_4model_100_summary_*.csv vào:\n",
        ANDROID_RESULT_DIR
    )

    android_summary = None

else:

    android_summary_path = summary_files[-1]

    android_summary = pd.read_csv(
        android_summary_path
    )

    print(
        "Using:",
        android_summary_path
    )

    display(
        android_summary
    )


## 9. MASTER TABLE — 4 model PC + Android

Đây là bảng chính để so trade-off.

- Accuracy chính thức: full test set.
- `pc100_*`: cùng subset 100 ảnh.
- Android: performance thực tế trên Galaxy A05s.
- Mean confidence chỉ là confidence, **không phải accuracy**.


In [ ]:
if android_summary is not None:

    master = (
        full_test
        .merge(
            pc100,
            on="model",
            how="left"
        )
        .merge(
            android_summary,
            on="model",
            how="left"
        )
    )

    important_columns = [
        "model",
        "input_size",

        "full_precision",
        "full_recall",
        "full_mAP50",
        "full_mAP50_95",

        "pc100_precision",
        "pc100_recall",
        "pc100_mAP50",
        "pc100_mAP50_95",

        "mean_conf_detected_only",
        "detection_rate_percent",

        "inference_mean_ms",
        "inference_median_ms",
        "inference_p95_ms",

        "total_median_ms",
        "total_p95_ms",
        "estimated_fps",

        "cpu_mean_percent",
        "cpu_p95_percent",

        "pss_mean_mb",
        "pss_peak_mb",

        "rss_mean_mb",
        "rss_peak_mb",

        "model_size_mb",
        "delegate",
    ]

    available_columns = [
        c for c in important_columns
        if c in master.columns
    ]

    display(
        master[available_columns]
        .sort_values(
            "full_mAP50_95",
            ascending=False
        )
        .reset_index(drop=True)
    )

    master_file = (
        BENCHMARK_ROOT /
        "MASTER_4MODEL_PC_ANDROID_COMPARISON.csv"
    )

    master.to_csv(
        master_file,
        index=False,
        encoding="utf-8-sig"
    )

    print(
        "Saved:",
        master_file
    )


## 10. Pareto sơ bộ: Accuracy ↔ Android latency

Model nằm gần góc trên-trái sẽ hấp dẫn hơn:
- mAP50-95 cao;
- latency thấp.


In [ ]:
import matplotlib.pyplot as plt

if android_summary is not None:

    plot_df = master.dropna(
        subset=[
            "full_mAP50_95",
            "total_median_ms"
        ]
    )

    fig, ax = plt.subplots(
        figsize=(8, 5)
    )

    ax.scatter(
        plot_df["total_median_ms"],
        plot_df["full_mAP50_95"]
    )

    for _, row in plot_df.iterrows():

        ax.annotate(
            row["model"],
            (
                row["total_median_ms"],
                row["full_mAP50_95"]
            ),
            xytext=(5, 5),
            textcoords="offset points"
        )

    ax.set_xlabel(
        "Android median total latency (ms) — lower is better"
    )

    ax.set_ylabel(
        "Full-test TFLite mAP50-95 — higher is better"
    )

    ax.set_title(
        "4-Model Mobile Trade-off"
    )

    ax.grid(
        True,
        alpha=0.3
    )

    plt.show()


# Kết quả cần giữ sau notebook

Sau khi hoàn tất bạn sẽ có:

```text
android_benchmark_100/
├── benchmark_100_manifest.csv
├── pc_100image_4model_metrics.csv
├── pc_subset_dataset/
├── android_results/
│   ├── android_4model_100_summary_....csv
│   ├── android_4model_100_detail_....csv
│   └── android_4model_100_detections_....csv
└── MASTER_4MODEL_PC_ANDROID_COMPARISON.csv
```

Đây sẽ là bộ dữ liệu nền để làm:
- 4-model FP32 trade-off;
- chọn model deploy;
- sau đó mới thử FP32 vs INT8 / reduced precision trên model thắng.
